In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch10-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "metric.py"

# 同一批 BACE 分子，兩種評估問題
第一次需連網下載 MoleculeNet 的原始資料。中間一格把分析寫成暫存程式，最後一格執行並顯示圖。要做小練習，可以把自己的程式接在中間那格最下方，再執行這兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

class MorganBits1024(dc.feat.MolecularFeaturizer):
    def _featurize(self, mol, **kwargs):
        return generator.GetFingerprintAsNumPy(mol)

options = dict(featurizer=MorganBits1024(), transformers=[], data_dir=str(cache), save_dir=str(cache))
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay)

reg_tasks, reg_sets, _ = dc.molnet.load_bace_regression(splitter="scaffold", **options)
cls_tasks, cls_sets, _ = dc.molnet.load_bace_classification(splitter="scaffold", **options)
assert reg_tasks == ["pIC50"] and cls_tasks == ["Class"]
for r, c in zip(reg_sets, cls_sets):
    assert np.array_equal(r.ids, c.ids) and np.all(r.w > 0) and np.all(c.w > 0)
train_r, valid_r, test_r = reg_sets
train_c, valid_c, test_c = cls_sets
print("train, valid, test:", *(len(d) for d in reg_sets))
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1)
rf.fit(train_r.X, train_r.y.ravel())
obs = test_r.y.ravel()
pred = rf.predict(test_r.X)
print("MAE, RMSE, R2:", *(f"{v:.3f}" for v in
      (mean_absolute_error(obs, pred), root_mean_squared_error(obs, pred), r2_score(obs, pred))))
baseline = np.full_like(obs, np.median(train_r.y))
print("constant median MAE:", f"{mean_absolute_error(obs, baseline):.3f}")

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
clf.fit(train_c.X, train_c.y.ravel())
score = clf.predict_proba(test_c.X)[:, list(clf.classes_).index(1)]
label = test_c.y.ravel().astype(int)
picked = score >= 0.50
matrix = confusion_matrix(label, picked, labels=[0, 1])
print("ROC-AUC, AP:", f"{roc_auc_score(label, score):.3f}", f"{average_precision_score(label, score):.3f}")
print("TN, FP, FN, TP:", *matrix.ravel())

fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), layout="constrained")
axes[0].scatter(obs, pred, s=18, alpha=0.65, color="#276983")
low, high = min(obs.min(), pred.min()), max(obs.max(), pred.max())
axes[0].plot([low, high], [low, high], "--", color="#a84e24")
axes[0].set(aspect="equal", xlabel="Measured pIC50", ylabel="Predicted pIC50",
            title="BACE regression")
ConfusionMatrixDisplay(matrix, display_labels=["Inactive", "Active"]).plot(
    ax=axes[1], colorbar=False, cmap="Blues", values_format="d")
axes[1].set_title("BACE classification (cutoff 0.50)")
fig.savefig(figure_dir / "10-02-bace-metrics.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "10-02-bace-metrics.png")))